# 04 · Preparación Silver

| | |
|---|---|
| **Fase CRISP-DM** | 3 — Preparación de los datos |
| **Objetivo** | Aplicar las reglas de calidad R1–R9 del notebook 03, generar la máscara de calidad por celda y fecha, detectar los saltos de nivel y fijar el split temporal 80/20. |
| **Entradas** | `data/bronze/voltajes_EL_A.parquet`, `data/bronze/voltajes_EL_B.parquet` |
| **Salidas** | `data/silver/voltajes_silver.parquet`, `data/silver/saltos_nivel.parquet`, `data/silver/resumen_calidad.parquet`, `data/silver/split.json`, `data/silver/_metadata.json`, figuras en `reports/silver/` |

**Reglas aplicadas** (implementadas en `src/voltcast/cleaning.py`)

| # | Regla |
|---|---|
| R1 | Columnas renombradas a `A_V_total`, `A_kA`, `A_c001..A_c181` (ídem `B_`); se descarta `Average`. |
| R2 | `Date` → fecha calendario; en fechas duplicadas se conserva el último registro del día; frecuencia diaria continua. |
| R3 | Huecos de hasta `MAX_INTERP_DAYS` días → interpolación lineal (`interpolado`); los más largos quedan `NaN` (`hueco`). |
| R4 | `paro`: kA < `KA_SHUTDOWN`. El voltaje de ese día no es una observación válida: queda `NaN` o, si el paro dura ≤ `MAX_INTERP_DAYS` días, se interpola para dar continuidad a las ventanas de entrada. El código sigue siendo `paro`, así que nunca entra en las métricas. |
| R5 | `fuera_servicio`: tramos de al menos `OOS_MIN_DAYS` días con V < 0,5 V (o negativo) y kA normal, **o** sin dato mientras el resto de las celdas sí lo tiene. Se unen interrupciones de hasta `OOS_MERGE_DAYS` días y los paros o días sin registro no cortan el tramo. |
| R6 | `atipico`: V > 5 V o V < 0,5 V aislados con kA normal → `NaN` (interpolable como un hueco). |
| R7 | Se guarda `n_en_servicio` (celdas no `fuera_servicio` ese día) para la covariable `V_total / n_en_servicio` de Gold. |
| R8 | Split temporal 80/20 sobre el calendario diario, guardado en `split.json` para todas las capas siguientes. |
| R9 | Tabla de saltos de nivel por celda. **Solo para diagnóstico** en la evaluación; nunca es una entrada del modelo. |

**Formato de salida (largo):** `fecha, electrolizador, celda, voltaje, kA, V_total, n_en_servicio, calidad`.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap

from voltcast import cleaning as cl
from voltcast import config, io

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "silver"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
ELS = list(config.SHEETS)


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")


print(f"KA_SHUTDOWN={config.KA_SHUTDOWN} | MAX_INTERP_DAYS={config.MAX_INTERP_DAYS} | "
      f"OOS_MIN_DAYS={config.OOS_MIN_DAYS} | OOS_MERGE_DAYS={config.OOS_MERGE_DAYS} | "
      f"STEP_THRESHOLD_V={config.STEP_THRESHOLD_V} | VAL_FRACTION={config.VAL_FRACTION}")

## 1. Lectura de Bronze

In [ ]:
bronze = {el: io.read_bronze(el) for el in ELS}
schemas = {el: io.detect_schema(df) for el, df in bronze.items()}
bronze_meta = io.read_metadata("bronze")
for el in ELS:
    m = bronze_meta[io.bronze_name(el)]
    print(f"EL {el}: {len(bronze[el])} filas, {schemas[el].n_cells} celdas | origen {m['source_file']} "
          f"(sha256 {m['source_sha256'][:12]}…), ingerido {m['executed_at']}")

## 2. R1–R2: fecha calendario, duplicados y frecuencia diaria

In [ ]:
calendario = cl.calendar_from(*bronze.values())
wide = {el: cl.daily_wide(bronze[el], el, schemas[el].cell_cols, calendario) for el in ELS}

for el in ELS:
    dias = bronze[el]["Date"].dt.normalize()
    print(f"EL {el}: {len(bronze[el])} registros → {dias.nunique()} días únicos "
          f"({dias.duplicated().sum()} duplicados descartados) → {len(wide[el])} días calendario "
          f"({len(wide[el]) - dias.nunique()} días sin registro agregados como NaN)")
wide["A"].iloc[:3, :6]

## 3. R3–R7: máscara de calidad, interpolación y celdas en servicio

In [ ]:
partes, saltos = [], []
for el in ELS:
    largo, s = cl.silver_electrolizador(wide[el], el)
    partes.append(largo)
    saltos.append(s)
silver = cl.finalize_long(partes)
saltos = pd.concat(saltos, ignore_index=True).sort_values(["electrolizador", "celda", "fecha"], ignore_index=True)
print(f"Silver: {len(silver):,} filas ({silver['celda'].nunique()} celdas × {len(calendario)} días)")
silver.head()

In [ ]:
resumen = (silver.groupby(["electrolizador", "calidad"], observed=False).size().unstack("electrolizador")
           .rename_axis(columns=None))
resumen_pct = resumen.div(resumen.sum()).mul(100).round(2)
pd.concat({"celdas×día": resumen, "%": resumen_pct}, axis=1)

In [ ]:
# Tramos fuera de servicio detectados
oos = (silver[silver["calidad"] == "fuera_servicio"]
       .groupby(["electrolizador", "celda"], observed=True)["fecha"].agg(desde="min", hasta="max", dias="count"))
oos

In [ ]:
# Celdas en servicio por día
n_srv = silver.groupby(["fecha", "electrolizador"], observed=True)["n_en_servicio"].first().unstack()
fig, ax = plt.subplots(figsize=(14, 3))
for el in ELS:
    ax.plot(n_srv.index, n_srv[el], label=f"EL {el}")
ax.set(ylabel="celdas", title="Celdas en servicio (no fuera_servicio)"); ax.legend()
fig.tight_layout(); savefig(fig, "01_celdas_en_servicio")

In [ ]:
# Mapa de calidad celdas × tiempo (código más frecuente de cada semana)
COLORES = {"ok": "#e8f0e8", "interpolado": "#74c476", "hueco": "#bdbdbd",
           "atipico": "#fdae61", "paro": "#6a51a3", "fuera_servicio": "#d7301f"}
CMAP = ListedColormap([COLORES[c] for c in cl.CALIDAD])

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for ax, el in zip(axes, ELS):
    m = silver[silver["electrolizador"] == el].pivot(index="fecha", columns="celda", values="calidad")
    m = m[[c for c in m.columns if c.startswith(el)]].apply(lambda s: s.cat.codes)
    sem = m.resample("W").agg(lambda s: s.mode().iloc[0])
    ax.imshow(sem.T.to_numpy(), aspect="auto", cmap=CMAP, vmin=-0.5, vmax=len(cl.CALIDAD) - 0.5,
              interpolation="nearest",
              extent=[mdates.date2num(sem.index[0]), mdates.date2num(sem.index[-1]), sem.shape[1] + 0.5, 0.5])
    ax.xaxis_date(); ax.set(ylabel="celda", title=f"EL {el} — calidad Silver (semanal)")
handles = [plt.Rectangle((0, 0), 1, 1, color=COLORES[c]) for c in cl.CALIDAD]
fig.legend(handles, cl.CALIDAD, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(); savefig(fig, "02_mapa_calidad")

In [ ]:
# Bronze frente a Silver en celdas con distintos casos
ejemplos = ["A_c061", "B_c046", "B_c080", "A_c100"]
fig, axes = plt.subplots(len(ejemplos), 1, figsize=(14, 2.6 * len(ejemplos)), sharex=True)
for ax, c in zip(axes, ejemplos):
    el = c[0]
    raw = wide[el][c]
    s = silver[silver["celda"] == c].set_index("fecha")
    ax.plot(raw.index, raw, color="lightgrey", lw=0.8, label="Bronze")
    ax.plot(s.index, s["voltaje"], color="k", lw=0.8, label="Silver")
    for cod in ["interpolado", "paro", "fuera_servicio"]:
        f = s.index[s["calidad"] == cod]
        ax.scatter(f, np.full(len(f), 0.2), s=4, color=COLORES[cod], label=cod)
    ax.set(ylim=(0, 4), ylabel="V", title=c); ax.legend(loc="lower right", ncol=5, fontsize=8)
fig.tight_layout(); savefig(fig, "03_bronze_vs_silver")

## 4. R9: saltos de nivel (solo diagnóstico)

In [ ]:
saltos["tipo"] = np.where(saltos["delta_V"] < 0, "caída", "subida")
print(f"{len(saltos)} saltos |ΔV| > {config.STEP_THRESHOLD_V} V en {saltos['celda'].nunique()} celdas")
saltos.groupby(["electrolizador", "tipo"]).agg(saltos=("delta_V", "size"), celdas=("celda", "nunique"),
                                               delta_mediano=("delta_V", "median")).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"width_ratios": [3, 1]})
for el, color in zip(ELS, ["tab:blue", "tab:orange"]):
    s = saltos[saltos["electrolizador"] == el]
    semana = s.set_index("fecha").resample("W")["delta_V"].size()
    axes[0].bar(semana.index, semana.values, width=5, alpha=0.7, color=color, label=f"EL {el}")
    axes[1].hist(s["delta_V"], bins=40, alpha=0.6, color=color, label=f"EL {el}")
axes[0].set(ylabel="saltos por semana", title="Saltos de nivel por celda"); axes[0].legend()
axes[1].set(xlabel="ΔV (V)", title="Magnitud"); axes[1].legend()
fig.tight_layout(); savefig(fig, "04_saltos_nivel")

## 5. R8: split temporal 80/20

In [ ]:
split = cl.temporal_split(calendario, config.VAL_FRACTION)
print(f"El {config.VAL_FRACTION:.0%} de validación equivale a {split['n_val']} filas diarias, "
      f"abarcando desde {split['val_desde']} hasta {split['val_hasta']}.")
print(f"Train: {split['n_train']} días, desde {split['train_desde']} hasta {split['train_hasta']}.")

val = silver["fecha"] >= split["fecha_corte"]
pd.crosstab(silver["calidad"], np.where(val, "validación", "train"), normalize="columns").mul(100).round(2)

In [ ]:
# Saltos de nivel dentro de la validación: los orígenes cuyo horizonte los cruce se reportarán aparte (notebook 07)
sv = saltos[saltos["fecha"] >= split["fecha_corte"]]
print(f"Saltos en validación: {len(sv)} en {sv['celda'].nunique()} celdas")
sv.assign(mes=sv["fecha"].dt.to_period("M")).groupby(["mes", "electrolizador"]).size().unstack(fill_value=0)

## 6. Escritura en Silver

In [ ]:
fuentes = {io.bronze_name(el): bronze_meta[io.bronze_name(el)]["source_sha256"] for el in ELS}
bronze_hash = {name: io.file_sha256(io.layer_path("bronze", name)) for name in fuentes}
comun = {"bronze_sha256": bronze_hash, "raw_sha256": fuentes}
params = {k: getattr(config, k) for k in ["KA_SHUTDOWN", "MAX_INTERP_DAYS", "OOS_MIN_DAYS", "OOS_MERGE_DAYS",
                                           "V_OUT_OF_SERVICE", "V_MAX_PLAUSIBLE", "STEP_THRESHOLD_V",
                                           "STEP_WINDOW_DAYS", "VAL_FRACTION"]}

resumen_out = resumen.reset_index().melt(id_vars="calidad", var_name="electrolizador", value_name="celdas_dia")
salidas = [
    (silver, "voltajes_silver.parquet", {"params": params, "split": split}),
    (saltos, "saltos_nivel.parquet", {"uso": "solo diagnóstico; no es entrada del modelo"}),
    (resumen_out, "resumen_calidad.parquet", {}),
]
for df, name, extra in salidas:
    p = io.write_layer(df, "silver", name, source=config.BRONZE_DIR, extra={**comun, **extra})
    print(f"✔ {p.relative_to(config.BASE_DIR)} ({len(df):,} filas, {p.stat().st_size / 1e6:.2f} MB)")
p = io.write_json(split, "silver", "split.json")
print(f"✔ {p.relative_to(config.BASE_DIR)}")

## 7. Verificaciones

In [ ]:
s = io.read_layer("silver", "voltajes_silver.parquet")
sp = io.read_json("silver", "split.json")
valido = s["calidad"].isin(cl.CALIDAD_VALIDA)

checks = {
    "una fila por (celda, fecha)": not s.duplicated(["celda", "fecha"]).any(),
    "calendario diario continuo": s.groupby("celda", observed=True)["fecha"].apply(
        lambda f: f.diff().dropna().eq(pd.Timedelta(days=1)).all()).all(),
    "celdas por electrolizador = 181": s.groupby("electrolizador", observed=True)["celda"].nunique().eq(181).all(),
    "ok/interpolado sin NaN": s.loc[valido, "voltaje"].notna().all(),
    "hueco y fuera_servicio son NaN": s.loc[s["calidad"].isin(["hueco", "fuera_servicio"]), "voltaje"].isna().all(),
    f"voltajes válidos en [{config.V_OUT_OF_SERVICE}, {config.V_MAX_PLAUSIBLE}] V": s.loc[valido, "voltaje"].between(
        config.V_OUT_OF_SERVICE, config.V_MAX_PLAUSIBLE).all(),
    "split sin solapamiento": pd.Timestamp(sp["train_hasta"]) < pd.Timestamp(sp["val_desde"]),
    "split cubre el calendario": sp["n_train"] + sp["n_val"] == s["fecha"].nunique(),
}
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values()), "Alguna verificación de Silver falló"

## Conclusión

- Silver contiene una serie diaria continua por celda (2 × 181) con su código de calidad. Solo las fechas `ok`/`interpolado` se usan como objetivo y en las métricas.
- Las celdas **61, 62, 123 y 124** de ambos electrolizadores quedan `fuera_servicio` desde 2020-01 hasta ≈ oct-2024 (EL A 123/124 hasta abr-2025), junto con la celda 46 y los tramos cortos del EL B. Esas muestras se excluyen en Gold y en el modelado.
- El split 80/20 queda fijado en `split.json`; todos los ajustes de las capas siguientes usan solo train.
- La tabla `saltos_nivel.parquet` se usará en el notebook 07 para separar el error de los horizontes que cruzan una intervención.

**Siguiente:** notebook 05 — BO-VMD y features (Gold).